In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install awscli --quiet
!mkdir -p /content/drive/MyDrive/chbmit
!aws s3 sync --no-sign-request s3://physionet-open/chbmit/1.0.0/ /content/drive/MyDrive/chbmit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 42.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [ ]:
!du -sh /content/drive/MyDrive/chbmit
!find /content/drive/MyDrive/chbmit -name '*.edf' | wc -l

du: cannot access '/content/drive/MyDrive/chbmit': Transport endpoint is not connected
find: ‘/content/drive/MyDrive/chbmit’: Transport endpoint is not connected
0


In [ ]:
!du -sh /content/drive/MyDrive/chbmit
!find /content/drive/MyDrive/chbmit -name '*.edf' | wc -l
!find /content/drive/MyDrive/chbmit -name '*.seizures' | wc -l

du: cannot access '/content/drive/MyDrive/chbmit': Transport endpoint is not connected
find: ‘/content/drive/MyDrive/chbmit’: Transport endpoint is not connected
0
find: ‘/content/drive/MyDrive/chbmit’: Transport endpoint is not connected
0


In [ ]:
# Get the official file list from PhysioNet
!wget -q -O /tmp/RECORDS https://physionet.org/files/chbmit/1.0.0/RECORDS
!wget -q -O /tmp/RECORDS-WITH-SEIZURES https://physionet.org/files/chbmit/1.0.0/RECORDS-WITH-SEIZURES

echo "Official RECORDS count:"
!wc -l /tmp/RECORDS

echo "Official RECORDS-WITH-SEIZURES count:"
!wc -l /tmp/RECORDS-WITH-SEIZURES

echo "Any .edf in your Drive not listed in the official RECORDS?"
!find /content/drive/MyDrive/chbmit -name '*.edf' | sed 's|.*chbmit/||' | sort > /tmp/local_edf.txt
!sort /tmp/RECORDS > /tmp/official_edf.txt
!comm -23 /tmp/local_edf.txt /tmp/official_edf.txt | head -20


SyntaxError: invalid syntax (2660530801.py, line 5)

In [ ]:
!wget -q -O /tmp/RECORDS https://physionet.org/files/chbmit/1.0.0/RECORDS
!wget -q -O /tmp/RECORDS-WITH-SEIZURES https://physionet.org/files/chbmit/1.0.0/RECORDS-WITH-SEIZURES

!echo "Official RECORDS count:"
!wc -l /tmp/RECORDS

!echo "Official RECORDS-WITH-SEIZURES count:"
!wc -l /tmp/RECORDS-WITH-SEIZURES

!echo "Any .edf in your Drive not listed in the official RECORDS?"
!find /content/drive/MyDrive/chbmit -name '*.edf' | sed 's|.*chbmit/||' | sort > /tmp/local_edf.txt
!sort /tmp/RECORDS > /tmp/official_edf.txt
!comm -23 /tmp/local_edf.txt /tmp/official_edf.txt | head -20

In [ ]:
import os
import re
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/chbmit"
records = []

for subject in sorted(os.listdir(DRIVE_ROOT)):
    subj_dir = os.path.join(DRIVE_ROOT, subject)
    summary_path = os.path.join(subj_dir, f"{subject}-summary.txt")
    if not os.path.isfile(summary_path):
        continue

    with open(summary_path, "r", errors="ignore") as f:
        text = f.read()

    # Split on each "File Name:" block
    blocks = re.split(r"\nFile Name:", text)[1:]
    for block in blocks:
        block = "File Name:" + block
        fname_match = re.search(r"File Name:\s*(\S+)", block)
        nseiz_match = re.search(r"Number of Seizures in File:\s*(\d+)", block)
        if not fname_match or not nseiz_match:
            continue

        fname = fname_match.group(1)
        n_seizures = int(nseiz_match.group(1))

        seizure_times = re.findall(
            r"Seizure(?:\s\d+)?\sStart Time:\s*(\d+)\s*seconds\s*"
            r"Seizure(?:\s\d+)?\sEnd Time:\s*(\d+)\s*seconds",
            block
        )

        if n_seizures == 0:
            records.append({
                "subject": subject, "file": fname,
                "has_seizure": 0, "seizure_start_sec": None, "seizure_end_sec": None
            })
        else:
            for start, end in seizure_times:
                records.append({
                    "subject": subject, "file": fname,
                    "has_seizure": 1,
                    "seizure_start_sec": int(start),
                    "seizure_end_sec": int(end)
                })

df = pd.DataFrame(records)
df.to_csv(os.path.join(DRIVE_ROOT, "master_annotations.csv"), index=False)
print(f"Parsed {df['subject'].nunique()} subjects, {len(df)} rows")
print(f"Total seizure events: {df['has_seizure'].sum()}")
df.head(10)

In [ ]:
!pip install pyedflib --quiet

In [ ]:
import numpy as np
import pyedflib

WINDOW_SEC = 4
OVERLAP = 0.5
STEP_SEC = WINDOW_SEC * (1 - OVERLAP)

def get_windows_for_file(subject, fname, seizure_intervals, fs=256):
    """seizure_intervals: list of (start_sec, end_sec) tuples for this file"""
    filepath = os.path.join(DRIVE_ROOT, subject, fname)
    if not os.path.isfile(filepath):
        return None, None

    f = pyedflib.EdfReader(filepath)
    n_channels = f.signals_in_file
    signal_len_sec = f.file_duration
    f.close()

    windows = []
    labels = []
    t = 0
    while t + WINDOW_SEC <= signal_len_sec:
        label = 0
        for s_start, s_end in seizure_intervals:
            # window overlaps a seizure if any part intersects
            if t < s_end and (t + WINDOW_SEC) > s_start:
                label = 1
                break
        windows.append((t, t + WINDOW_SEC))
        labels.append(label)
        t += STEP_SEC

    return windows, labels

In [ ]:
window_records = []

for subject in df["subject"].unique():
    subj_df = df[df["subject"] == subject]
    for fname in subj_df["file"].unique():
        file_rows = subj_df[subj_df["file"] == fname]
        seizure_intervals = [
            (row.seizure_start_sec, row.seizure_end_sec)
            for row in file_rows.itertuples()
            if row.has_seizure == 1
        ]
        windows, labels = get_windows_for_file(subject, fname, seizure_intervals)
        if windows is None:
            continue
        for (w_start, w_end), label in zip(windows, labels):
            window_records.append({
                "subject": subject, "file": fname,
                "window_start_sec": w_start, "window_end_sec": w_end,
                "label": label
            })

window_df = pd.DataFrame(window_records)
window_df.to_csv(os.path.join(DRIVE_ROOT, "window_index.csv"), index=False)
print(f"Total windows: {len(window_df)}")
print(f"Seizure windows: {window_df['label'].sum()} ({100*window_df['label'].mean():.2f}%)")

In [ ]:
import numpy as np

SEIZURE_TO_NONSEIZURE_RATIO = 15  # tune later

# Split seizure vs non-seizure windows
seizure_windows = window_df[window_df["label"] == 1]
nonseizure_windows = window_df[window_df["label"] == 0]

# Sample non-seizure windows for a balanced-ish training subset
n_nonseizure_needed = len(seizure_windows) * SEIZURE_TO_NONSEIZURE_RATIO
sampled_nonseizure = nonseizure_windows.sample(n=n_nonseizure_needed, random_state=42)

train_subset = pd.concat([seizure_windows, sampled_nonseizure]).sort_values(["subject", "file", "window_start_sec"])
print(f"Training subset size: {len(train_subset)} ({len(seizure_windows)} seizure + {len(sampled_nonseizure)} non-seizure)")

# Now extract actual signal data — ONE EDF read per file, not per window
def extract_signals_for_file(subject, fname, windows_needed, fs=256):
    filepath = os.path.join(DRIVE_ROOT, subject, fname)
    f = pyedflib.EdfReader(filepath)
    n_ch = f.signals_in_file
    signal = np.array([f.readSignal(i) for i in range(n_ch)])  # shape: (channels, total_samples)
    f.close()

    extracted = []
    for _, row in windows_needed.iterrows():
        start_sample = int(row.window_start_sec * fs)
        end_sample = int(row.window_end_sec * fs)
        extracted.append(signal[:, start_sample:end_sample])
    return extracted

# Process file-by-file, save each subject's shard separately
os.makedirs(os.path.join(DRIVE_ROOT, "shards"), exist_ok=True)

for (subject, fname), group in train_subset.groupby(["subject", "file"]):
    signals = extract_signals_for_file(subject, fname, group)
    labels = group["label"].values
    shard_name = f"{subject}_{fname.replace('.edf','')}.npz"
    np.savez_compressed(
        os.path.join(DRIVE_ROOT, "shards", shard_name),
        signals=np.array(signals), labels=labels
    )

print("Shard extraction complete.")

In [ ]:
import glob

shard_files = glob.glob(os.path.join(DRIVE_ROOT, "shards", "*.npz"))
print(f"Total shards: {len(shard_files)}")

# Group shards by subject for patient-level split
from collections import defaultdict
subject_shards = defaultdict(list)
for path in shard_files:
    subject = os.path.basename(path).split("_")[0]  # e.g. "chb01"
    subject_shards[subject].append(path)

subjects = sorted(subject_shards.keys())
print(f"Subjects with shards: {subjects}")

# Hold out ~20% of subjects entirely for subject-independent test set
np.random.seed(42)
shuffled = np.random.permutation(subjects)
n_test_subjects = max(1, int(0.2 * len(subjects)))
test_subjects = set(shuffled[:n_test_subjects])
train_subjects = set(shuffled[n_test_subjects:])

print(f"Train subjects ({len(train_subjects)}): {sorted(train_subjects)}")
print(f"Test subjects ({len(test_subjects)}): {sorted(test_subjects)}")

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import glob
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/chbmit"

# Reload the CSVs you already built — no need to re-parse anything
df = pd.read_csv(os.path.join(DRIVE_ROOT, "master_annotations.csv"))
window_df = pd.read_csv(os.path.join(DRIVE_ROOT, "window_index.csv"))

print(f"Reloaded: {len(df)} annotation rows, {len(window_df)} window rows")


In [ ]:
import glob

shard_files = glob.glob(os.path.join(DRIVE_ROOT, "shards", "*.npz"))
print(f"Total shards: {len(shard_files)}")

# Group shards by subject for patient-level split
from collections import defaultdict
subject_shards = defaultdict(list)
for path in shard_files:
    subject = os.path.basename(path).split("_")[0]  # e.g. "chb01"
    subject_shards[subject].append(path)

subjects = sorted(subject_shards.keys())
print(f"Subjects with shards: {subjects}")

# Hold out ~20% of subjects entirely for subject-independent test set
np.random.seed(42)
shuffled = np.random.permutation(subjects)
n_test_subjects = max(1, int(0.2 * len(subjects)))
test_subjects = set(shuffled[:n_test_subjects])
train_subjects = set(shuffled[n_test_subjects:])

print(f"Train subjects ({len(train_subjects)}): {sorted(train_subjects)}")
print(f"Test subjects ({len(test_subjects)}): {sorted(test_subjects)}")

In [ ]:
def load_shards(subject_list, subject_shards_map):
    X, y, shapes = [], [], set()
    for subject in subject_list:
        for path in subject_shards_map[subject]:
            data = np.load(path)
            signals = data["signals"]  # shape: (n_windows, n_channels, n_samples)
            labels = data["labels"]
            for sig in signals:
                X.append(sig)
                shapes.add(sig.shape)
            y.extend(labels)
    return X, np.array(y), shapes

X_train_raw, y_train, train_shapes = load_shards(train_subjects, subject_shards)
X_test_raw, y_test, test_shapes = load_shards(test_subjects, subject_shards)

print(f"Train: {len(X_train_raw)} windows, {y_train.sum()} seizure")
print(f"Test: {len(X_test_raw)} windows, {y_test.sum()} seizure")
print(f"Unique shapes in train: {train_shapes}")
print(f"Unique shapes in test: {test_shapes}")

In [ ]:
drive.mount('/content/drive', force_remount=True)

In [ ]:
def load_shards(subject_list, subject_shards_map):
    X, y, shapes = [], [], set()
    for subject in subject_list:
        for path in subject_shards_map[subject]:
            data = np.load(path)
            signals = data["signals"]  # shape: (n_windows, n_channels, n_samples)
            labels = data["labels"]
            for sig in signals:
                X.append(sig)
                shapes.add(sig.shape)
            y.extend(labels)
    return X, np.array(y), shapes

X_train_raw, y_train, train_shapes = load_shards(train_subjects, subject_shards)
X_test_raw, y_test, test_shapes = load_shards(test_subjects, subject_shards)

print(f"Train: {len(X_train_raw)} windows, {y_train.sum()} seizure")
print(f"Test: {len(X_test_raw)} windows, {y_test.sum()} seizure")
print(f"Unique shapes in train: {train_shapes}")
print(f"Unique shapes in test: {test_shapes}")

In [ ]:
import shutil

LOCAL_SHARD_DIR = "/content/local_shards"
os.makedirs(LOCAL_SHARD_DIR, exist_ok=True)

drive_shard_dir = os.path.join(DRIVE_ROOT, "shards")
for fname in os.listdir(drive_shard_dir):
    src = os.path.join(drive_shard_dir, fname)
    dst = os.path.join(LOCAL_SHARD_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

print(f"Copied {len(os.listdir(LOCAL_SHARD_DIR))} shards to local disk")

In [ ]:
import os
import pandas as pd
import numpy as np
import glob
import shutil
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/chbmit"

# Reload annotation/window CSVs
df = pd.read_csv(os.path.join(DRIVE_ROOT, "master_annotations.csv"))
window_df = pd.read_csv(os.path.join(DRIVE_ROOT, "window_index.csv"))

# Copy shards to local disk (fast if already copied — skips existing files)
LOCAL_SHARD_DIR = "/content/local_shards"
os.makedirs(LOCAL_SHARD_DIR, exist_ok=True)
drive_shard_dir = os.path.join(DRIVE_ROOT, "shards")
for fname in os.listdir(drive_shard_dir):
    src = os.path.join(drive_shard_dir, fname)
    dst = os.path.join(LOCAL_SHARD_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

# Rebuild subject -> shard path mapping from LOCAL copies
subject_shards = defaultdict(list)
for fname in os.listdir(LOCAL_SHARD_DIR):
    subject = fname.split("_")[0]
    subject_shards[subject].append(os.path.join(LOCAL_SHARD_DIR, fname))

subjects = sorted(subject_shards.keys())

# Recreate the SAME train/test split (fixed seed = same result every time)
np.random.seed(42)
shuffled = np.random.permutation(subjects)
n_test_subjects = max(1, int(0.2 * len(subjects)))
test_subjects = set(shuffled[:n_test_subjects])
train_subjects = set(shuffled[n_test_subjects:])

print(f"Ready: {len(subjects)} subjects, {sum(len(v) for v in subject_shards.values())} shards")
print(f"Train subjects ({len(train_subjects)}): {sorted(train_subjects)}")
print(f"Test subjects ({len(test_subjects)}): {sorted(test_subjects)}")

In [ ]:
def load_shards(subject_list, subject_shards_map):
    X, y, shapes = [], [], set()
    for subject in subject_list:
        for path in subject_shards_map[subject]:
            data = np.load(path)
            signals = data["signals"]  # shape: (n_windows, n_channels, n_samples)
            labels = data["labels"]
            for sig in signals:
                X.append(sig)
                shapes.add(sig.shape)
            y.extend(labels)
    return X, np.array(y), shapes

X_train_raw, y_train, train_shapes = load_shards(train_subjects, subject_shards)
X_test_raw, y_test, test_shapes = load_shards(test_subjects, subject_shards)

print(f"Train: {len(X_train_raw)} windows, {y_train.sum()} seizure")
print(f"Test: {len(X_test_raw)} windows, {y_test.sum()} seizure")
print(f"Unique shapes in train: {train_shapes}")
print(f"Unique shapes in test: {test_shapes}")

In [ ]:
import tensorflow as tf

def shard_generator(subject_list, subject_shards_map):
    for subject in subject_list:
        for path in subject_shards_map[subject]:
            data = np.load(path)
            signals = data["signals"].astype(np.float32)  # float32, not float64 — halves memory
            labels = data["labels"].astype(np.float32)
            for sig, lbl in zip(signals, labels):
                yield sig, lbl

# First, check shape consistency cheaply — WITHOUT loading everything
def check_shapes(subject_list, subject_shards_map, max_check=50):
    shapes = set()
    count = 0
    for subject in subject_list:
        for path in subject_shards_map[subject]:
            data = np.load(path)
            shapes.add(data["signals"].shape[1:])  # (channels, samples) per window
            count += 1
            if count >= max_check:
                return shapes
    return shapes

train_shapes = check_shapes(list(train_subjects), subject_shards)
print(f"Sample of shapes seen in train shards: {train_shapes}")

In [ ]:
import os
import pandas as pd
import numpy as np
import glob
import shutil
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/chbmit"

df = pd.read_csv(os.path.join(DRIVE_ROOT, "master_annotations.csv"))
window_df = pd.read_csv(os.path.join(DRIVE_ROOT, "window_index.csv"))

LOCAL_SHARD_DIR = "/content/local_shards"
os.makedirs(LOCAL_SHARD_DIR, exist_ok=True)
drive_shard_dir = os.path.join(DRIVE_ROOT, "shards")
for fname in os.listdir(drive_shard_dir):
    src = os.path.join(drive_shard_dir, fname)
    dst = os.path.join(LOCAL_SHARD_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

subject_shards = defaultdict(list)
for fname in os.listdir(LOCAL_SHARD_DIR):
    subject = fname.split("_")[0]
    subject_shards[subject].append(os.path.join(LOCAL_SHARD_DIR, fname))

subjects = sorted(subject_shards.keys())
np.random.seed(42)
shuffled = np.random.permutation(subjects)
n_test_subjects = max(1, int(0.2 * len(subjects)))
test_subjects = set(shuffled[:n_test_subjects])
train_subjects = set(shuffled[n_test_subjects:])

print(f"Ready: {len(subjects)} subjects, {sum(len(v) for v in subject_shards.values())} shards")

In [ ]:
def check_shapes(subject_list, subject_shards_map, max_check=50):
    shapes = set()
    count = 0
    for subject in subject_list:
        for path in subject_shards_map[subject]:
            data = np.load(path)
            shapes.add(data["signals"].shape[1:])
            count += 1
            if count >= max_check:
                return shapes
    return shapes

train_shapes = check_shapes(list(train_subjects), subject_shards)
print(f"Sample of shapes seen in train shards: {train_shapes}")

In [ ]:
def extract_signals_for_file_by_name(subject, fname, windows_needed, fs=256):
    filepath = os.path.join(DRIVE_ROOT, subject, fname)
    f = pyedflib.EdfReader(filepath)
    labels = f.getSignalLabels()

    # Find indices of our common channels, in COMMON_CHANNELS order
    label_to_idx = {lbl: i for i, lbl in enumerate(labels)}
    if not all(ch in label_to_idx for ch in COMMON_CHANNELS):
        f.close()
        return None  # skip files missing any required channel

    chosen_idx = [label_to_idx[ch] for ch in COMMON_CHANNELS]
    full_signal = np.array([f.readSignal(i) for i in chosen_idx])  # shape: (18, total_samples)
    f.close()

    extracted = []
    for _, row in windows_needed.iterrows():
        start_sample = int(row.window_start_sec * fs)
        end_sample = int(row.window_end_sec * fs)
        extracted.append(full_signal[:, start_sample:end_sample])
    return extracted

# Re-run shard extraction with the fixed function, skipping incompatible files
skipped_files = []
for (subject, fname), group in train_subset.groupby(["subject", "file"]):
    signals = extract_signals_for_file_by_name(subject, fname, group)
    if signals is None:
        skipped_files.append((subject, fname))
        continue
    labels_arr = group["label"].values
    shard_name = f"{subject}_{fname.replace('.edf','')}.npz"
    np.savez_compressed(
        os.path.join(DRIVE_ROOT, "shards", shard_name),
        signals=np.array(signals), labels=labels_arr
    )

print(f"Re-extraction complete. Skipped {len(skipped_files)} files missing required channels.")
print(skipped_files[:20])

In [ ]:
SEIZURE_TO_NONSEIZURE_RATIO = 15

seizure_windows = window_df[window_df["label"] == 1] if "label" in window_df.columns else window_df[window_df["has_seizure"] == 1]
nonseizure_windows = window_df[window_df["label"] == 0] if "label" in window_df.columns else window_df[window_df["has_seizure"] == 0]

n_nonseizure_needed = len(seizure_windows) * SEIZURE_TO_NONSEIZURE_RATIO
sampled_nonseizure = nonseizure_windows.sample(n=n_nonseizure_needed, random_state=42)

train_subset = pd.concat([seizure_windows, sampled_nonseizure]).sort_values(["subject", "file"])
print(f"Rebuilt training subset: {len(train_subset)} ({len(seizure_windows)} seizure + {len(sampled_nonseizure)} non-seizure)")

# SAVE IT so this never has to be rebuilt from scratch again
train_subset.to_csv(os.path.join(DRIVE_ROOT, "train_subset.csv"), index=False)

In [ ]:
COMMON_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ"
]

def extract_signals_for_file_by_name(subject, fname, windows_needed, fs=256):
    filepath = os.path.join(DRIVE_ROOT, subject, fname)
    f = pyedflib.EdfReader(filepath)
    labels = f.getSignalLabels()

    label_to_idx = {lbl: i for i, lbl in enumerate(labels)}
    if not all(ch in label_to_idx for ch in COMMON_CHANNELS):
        f.close()
        return None

    chosen_idx = [label_to_idx[ch] for ch in COMMON_CHANNELS]
    full_signal = np.array([f.readSignal(i) for i in chosen_idx])
    f.close()

    extracted = []
    for _, row in windows_needed.iterrows():
        start_sample = int(row.window_start_sec * fs)
        end_sample = int(row.window_end_sec * fs)
        extracted.append(full_signal[:, start_sample:end_sample])
    return extracted

skipped_files = []
processed = 0
for (subject, fname), group in train_subset.groupby(["subject", "file"]):
    signals = extract_signals_for_file_by_name(subject, fname, group)
    if signals is None:
        skipped_files.append((subject, fname))
        continue
    labels_arr = group["label"].values if "label" in group.columns else group["has_seizure"].values
    shard_name = f"{subject}_{fname.replace('.edf','')}.npz"
    np.savez_compressed(
        os.path.join(DRIVE_ROOT, "shards", shard_name),
        signals=np.array(signals), labels=labels_arr
    )
    processed += 1

print(f"Re-extraction complete. Processed {processed} files, skipped {len(skipped_files)}.")
print("Skipped files:", skipped_files[:20])

In [ ]:
!pip install pyedflib --quiet

In [ ]:
COMMON_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ"
]

def extract_signals_for_file_by_name(subject, fname, windows_needed, fs=256):
    filepath = os.path.join(DRIVE_ROOT, subject, fname)
    f = pyedflib.EdfReader(filepath)
    labels = f.getSignalLabels()

    label_to_idx = {lbl: i for i, lbl in enumerate(labels)}
    if not all(ch in label_to_idx for ch in COMMON_CHANNELS):
        f.close()
        return None

    chosen_idx = [label_to_idx[ch] for ch in COMMON_CHANNELS]
    full_signal = np.array([f.readSignal(i) for i in chosen_idx])
    f.close()

    extracted = []
    for _, row in windows_needed.iterrows():
        start_sample = int(row.window_start_sec * fs)
        end_sample = int(row.window_end_sec * fs)
        extracted.append(full_signal[:, start_sample:end_sample])
    return extracted

skipped_files = []
processed = 0
for (subject, fname), group in train_subset.groupby(["subject", "file"]):
    signals = extract_signals_for_file_by_name(subject, fname, group)
    if signals is None:
        skipped_files.append((subject, fname))
        continue
    labels_arr = group["label"].values if "label" in group.columns else group["has_seizure"].values
    shard_name = f"{subject}_{fname.replace('.edf','')}.npz"
    np.savez_compressed(
        os.path.join(DRIVE_ROOT, "shards", shard_name),
        signals=np.array(signals), labels=labels_arr
    )
    processed += 1

print(f"Re-extraction complete. Processed {processed} files, skipped {len(skipped_files)}.")
print("Skipped files:", skipped_files[:20])

In [ ]:
!pip install pyedflib --quiet
import pyedflib

In [ ]:
COMMON_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ"
]

def extract_signals_for_file_by_name(subject, fname, windows_needed, fs=256):
    filepath = os.path.join(DRIVE_ROOT, subject, fname)
    f = pyedflib.EdfReader(filepath)
    labels = f.getSignalLabels()

    label_to_idx = {lbl: i for i, lbl in enumerate(labels)}
    if not all(ch in label_to_idx for ch in COMMON_CHANNELS):
        f.close()
        return None

    chosen_idx = [label_to_idx[ch] for ch in COMMON_CHANNELS]
    full_signal = np.array([f.readSignal(i) for i in chosen_idx])
    f.close()

    extracted = []
    for _, row in windows_needed.iterrows():
        start_sample = int(row.window_start_sec * fs)
        end_sample = int(row.window_end_sec * fs)
        extracted.append(full_signal[:, start_sample:end_sample])
    return extracted

skipped_files = []
processed = 0
for (subject, fname), group in train_subset.groupby(["subject", "file"]):
    signals = extract_signals_for_file_by_name(subject, fname, group)
    if signals is None:
        skipped_files.append((subject, fname))
        continue
    labels_arr = group["label"].values if "label" in group.columns else group["has_seizure"].values
    shard_name = f"{subject}_{fname.replace('.edf','')}.npz"
    np.savez_compressed(
        os.path.join(DRIVE_ROOT, "shards", shard_name),
        signals=np.array(signals), labels=labels_arr
    )
    processed += 1

print(f"Re-extraction complete. Processed {processed} files, skipped {len(skipped_files)}.")
print("Skipped files:", skipped_files[:20])

In [ ]:
# Re-copy updated shards to local disk (overwrite old mismatched-shape ones)
for fname in os.listdir(drive_shard_dir):
    src = os.path.join(drive_shard_dir, fname)
    dst = os.path.join(LOCAL_SHARD_DIR, fname)
    shutil.copy(src, dst)  # no existence check this time — force overwrite

# Rebuild the subject->shard mapping fresh
subject_shards = defaultdict(list)
for fname in os.listdir(LOCAL_SHARD_DIR):
    subject = fname.split("_")[0]
    subject_shards[subject].append(os.path.join(LOCAL_SHARD_DIR, fname))

# Re-check shapes
train_shapes = check_shapes(list(train_subjects), subject_shards)
print(f"Shapes after fix: {train_shapes}")

In [ ]:
drive.mount('/content/drive', force_remount=True)

In [ ]:
import time

failed_copies = []

for fname in os.listdir(drive_shard_dir):
    src = os.path.join(drive_shard_dir, fname)
    dst = os.path.join(LOCAL_SHARD_DIR, fname)
    for attempt in range(2):
        try:
            shutil.copy(src, dst)
            break
        except OSError:
            if attempt == 0:
                time.sleep(1)  # brief pause, Drive mount often recovers
                continue
            failed_copies.append(fname)

print(f"Copy complete. Failed: {len(failed_copies)}")
print(failed_copies[:20])

In [ ]:
subject_shards = defaultdict(list)
for fname in os.listdir(LOCAL_SHARD_DIR):
    subject = fname.split("_")[0]
    subject_shards[subject].append(os.path.join(LOCAL_SHARD_DIR, fname))

train_shapes = check_shapes(list(train_subjects), subject_shards)
print(f"Shapes after fix: {train_shapes}")

In [ ]:
import tensorflow as tf

def make_dataset(subject_list, subject_shards_map, batch_size=64, shuffle=True):
    def gen():
        for subject in subject_list:
            for path in subject_shards_map[subject]:
                data = np.load(path)
                signals = data["signals"].astype(np.float32)  # (n_windows, 18, 1024)
                labels = data["labels"].astype(np.float32)
                for sig, lbl in zip(signals, labels):
                    yield sig, lbl

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(18, 1024), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=2000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(list(train_subjects), subject_shards, batch_size=64, shuffle=True)
test_ds = make_dataset(list(test_subjects), subject_shards, batch_size=64, shuffle=False)

# Quick sanity check — pull one batch, confirm shape
for x_batch, y_batch in train_ds.take(1):
    print(f"Batch X shape: {x_batch.shape}, Batch y shape: {y_batch.shape}")
    print(f"Batch label distribution: {y_batch.numpy().sum()} / {len(y_batch)} seizure")

In [ ]:
def build_model(input_shape=(18, 1024)):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2, 1))(inputs)  # (samples, channels) for Conv1D

    x = tf.keras.layers.SeparableConv1D(16, kernel_size=7, activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.SeparableConv1D(32, kernel_size=5, activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.SeparableConv1D(64, kernel_size=3, activation="relu", padding="same")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

# Class weighting to further help with the residual imbalance
# even after our 15:1 rebalancing, weight the positive class higher
class_weight = {0: 1.0, 1: 5.0}

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    class_weight=class_weight
)

In [ ]:
stale_files = [
    "chb12_chb12_27.npz",
    "chb12_chb12_28.npz",
    "chb12_chb12_29.npz"
]

for fname in stale_files:
    drive_path = os.path.join(drive_shard_dir, fname)
    local_path = os.path.join(LOCAL_SHARD_DIR, fname)
    if os.path.exists(drive_path):
        os.remove(drive_path)
        print(f"Deleted from Drive: {fname}")
    if os.path.exists(local_path):
        os.remove(local_path)
        print(f"Deleted from local: {fname}")

# Rebuild subject_shards mapping to reflect the deletion
subject_shards = defaultdict(list)
for fname in os.listdir(LOCAL_SHARD_DIR):
    subject = fname.split("_")[0]
    subject_shards[subject].append(os.path.join(LOCAL_SHARD_DIR, fname))

print(f"Total shards remaining: {sum(len(v) for v in subject_shards.values())}")

In [ ]:
all_shapes = check_shapes(list(train_subjects) + list(test_subjects), subject_shards, max_check=1000)
print(f"Shapes across all shards: {all_shapes}")

In [ ]:
train_ds = make_dataset(list(train_subjects), subject_shards, batch_size=64, shuffle=True)
test_ds = make_dataset(list(test_subjects), subject_shards, batch_size=64, shuffle=False)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    class_weight=class_weight
)

In [ ]:
print("Final training metrics:", {k: v[-1] for k, v in history.history.items() if not k.startswith("val_")})
print("Final validation metrics:", {k: v[-1] for k, v in history.history.items() if k.startswith("val_")})

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

y_true, y_pred_probs = [], []
for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch, verbose=0)
    y_true.extend(y_batch.numpy())
    y_pred_probs.extend(preds.flatten())

precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_probs)

In [ ]:
# Print precision/recall at a few specific thresholds
for target_thresh in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    idx = np.argmin(np.abs(thresholds - target_thresh))
    print(f"Threshold {target_thresh:.1f}: Precision={precisions[idx]:.3f}, Recall={recalls[idx]:.3f}")

In [ ]:
def smooth_predictions(probs, threshold=0.3, min_consecutive=3):
    """Only flag a seizure if min_consecutive consecutive windows exceed threshold"""
    binary = (np.array(probs) > threshold).astype(int)
    smoothed = np.zeros_like(binary)
    count = 0
    for i in range(len(binary)):
        if binary[i] == 1:
            count += 1
        else:
            count = 0
        if count >= min_consecutive:
            smoothed[max(0, i - min_consecutive + 1):i + 1] = 1
    return smoothed

y_pred_smoothed = smooth_predictions(y_pred_probs, threshold=0.3, min_consecutive=3)

from sklearn.metrics import precision_score, recall_score
print(f"Smoothed precision: {precision_score(y_true, y_pred_smoothed):.3f}")
print(f"Smoothed recall: {recall_score(y_true, y_pred_smoothed):.3f}")

In [ ]:
total_hours = len(y_true) * 4 / 3600
false_positives = np.sum((y_pred_smoothed == 1) & (np.array(y_true) == 0))
fa_per_hour = false_positives / total_hours
print(f"Total test hours: {total_hours:.1f}")
print(f"False alarms per hour: {fa_per_hour:.2f}")

In [ ]:
test_window_df = window_df[window_df["subject"].isin(test_subjects)]
print(f"Full test set: {len(test_window_df)} windows, {test_window_df['label'].sum() if 'label' in test_window_df.columns else test_window_df['has_seizure'].sum()} seizure")

os.makedirs(os.path.join(DRIVE_ROOT, "test_shards_full"), exist_ok=True)

skipped_test = []
processed_test = 0
for (subject, fname), group in test_window_df.groupby(["subject", "file"]):
    signals = extract_signals_for_file_by_name(subject, fname, group)
    if signals is None:
        skipped_test.append((subject, fname))
        continue
    labels_arr = group["label"].values if "label" in group.columns else group["has_seizure"].values
    shard_name = f"{subject}_{fname.replace('.edf','')}.npz"
    np.savez_compressed(
        os.path.join(DRIVE_ROOT, "test_shards_full", shard_name),
        signals=np.array(signals), labels=labels_arr
    )
    processed_test += 1

print(f"Full test extraction complete. Processed {processed_test}, skipped {len(skipped_test)}.")

In [ ]:
TEST_SHARD_DIR = os.path.join(DRIVE_ROOT, "test_shards_full")

# Copy to local disk first (same reliability fix as before)
LOCAL_TEST_SHARD_DIR = "/content/local_test_shards_full"
os.makedirs(LOCAL_TEST_SHARD_DIR, exist_ok=True)

for fname in os.listdir(TEST_SHARD_DIR):
    src = os.path.join(TEST_SHARD_DIR, fname)
    dst = os.path.join(LOCAL_TEST_SHARD_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

print(f"Copied {len(os.listdir(LOCAL_TEST_SHARD_DIR))} full test shards locally")

# Build subject->path map for the full test shards
full_test_shards = defaultdict(list)
for fname in os.listdir(LOCAL_TEST_SHARD_DIR):
    subject = fname.split("_")[0]
    full_test_shards[subject].append(os.path.join(LOCAL_TEST_SHARD_DIR, fname))

full_test_ds = make_dataset(list(test_subjects), full_test_shards, batch_size=64, shuffle=False)

In [ ]:
y_true_full, y_pred_probs_full = [], []
for x_batch, y_batch in full_test_ds:
    preds = model.predict(x_batch, verbose=0)
    y_true_full.extend(y_batch.numpy())
    y_pred_probs_full.extend(preds.flatten())

print(f"Total windows evaluated: {len(y_true_full)}")
print(f"Total seizure windows: {int(np.sum(y_true_full))}")

In [ ]:
from sklearn.metrics import precision_recall_curve, precision_score, recall_score

precisions_full, recalls_full, thresholds_full = precision_recall_curve(y_true_full, y_pred_probs_full)

for target_thresh in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    idx = np.argmin(np.abs(thresholds_full - target_thresh))
    print(f"Threshold {target_thresh:.1f}: Precision={precisions_full[idx]:.3f}, Recall={recalls_full[idx]:.3f}")

In [ ]:
total_hours_full = len(y_true_full) * 4 / 3600
print(f"Total test hours (full, honest): {total_hours_full:.1f}")

for min_c in [1, 3, 5, 8, 10, 15]:
    y_pred_smoothed_full = smooth_predictions(y_pred_probs_full, threshold=0.3, min_consecutive=min_c)
    p = precision_score(y_true_full, y_pred_smoothed_full)
    r = recall_score(y_true_full, y_pred_smoothed_full)
    fp = np.sum((y_pred_smoothed_full == 1) & (np.array(y_true_full) == 0))
    fa_hr = fp / total_hours_full
    print(f"min_consecutive={min_c}: Precision={p:.3f}, Recall={r:.3f}, FA/hr={fa_hr:.2f}")

In [ ]:
def normalize_window(window):
    # window shape: (channels, samples) — normalize per-channel
    mean = window.mean(axis=1, keepdims=True)
    std = window.std(axis=1, keepdims=True) + 1e-8
    return (window - mean) / std

In [ ]:
def make_dataset_normalized(subject_list, subject_shards_map, batch_size=64, shuffle=True):
    def gen():
        for subject in subject_list:
            for path in subject_shards_map[subject]:
                data = np.load(path)
                signals = data["signals"].astype(np.float32)
                labels = data["labels"].astype(np.float32)
                for sig, lbl in zip(signals, labels):
                    yield normalize_window(sig), lbl

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(18, 1024), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=2000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds_norm = make_dataset_normalized(list(train_subjects), subject_shards, batch_size=64, shuffle=True)
test_ds_norm = make_dataset_normalized(list(test_subjects), subject_shards, batch_size=64, shuffle=False)
full_test_ds_norm = make_dataset_normalized(list(test_subjects), full_test_shards, batch_size=64, shuffle=False)

# Fresh model, fresh weights — the input distribution has changed
model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

history = model.fit(
    train_ds_norm,
    validation_data=test_ds_norm,
    epochs=15,
    class_weight={0: 1.0, 1: 5.0}
)

In [ ]:
print("Final training metrics:", {k: v[-1] for k, v in history.history.items() if not k.startswith("val_")})
print("Final validation metrics:", {k: v[-1] for k, v in history.history.items() if k.startswith("val_")})

In [ ]:
y_true_full_norm, y_pred_probs_full_norm = [], []
for x_batch, y_batch in full_test_ds_norm:
    preds = model.predict(x_batch, verbose=0)
    y_true_full_norm.extend(y_batch.numpy())
    y_pred_probs_full_norm.extend(preds.flatten())

print(f"Total windows evaluated: {len(y_true_full_norm)}")

total_hours_full = len(y_true_full_norm) * 4 / 3600
for min_c in [1, 3, 5, 8, 10, 15]:
    y_pred_smoothed = smooth_predictions(y_pred_probs_full_norm, threshold=0.3, min_consecutive=min_c)
    p = precision_score(y_true_full_norm, y_pred_smoothed)
    r = recall_score(y_true_full_norm, y_pred_smoothed)
    fp = np.sum((y_pred_smoothed == 1) & (np.array(y_true_full_norm) == 0))
    fa_hr = fp / total_hours_full
    print(f"min_consecutive={min_c}: Precision={p:.3f}, Recall={r:.3f}, FA/hr={fa_hr:.2f}")

In [ ]:
def build_model_v2(input_shape=(18, 1024)):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2, 1))(inputs)

    x = tf.keras.layers.SeparableConv1D(32, kernel_size=7, activation="relu", padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)

    x = tf.keras.layers.SeparableConv1D(64, kernel_size=5, activation="relu", padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)

    x = tf.keras.layers.SeparableConv1D(128, kernel_size=3, activation="relu", padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)

    x = tf.keras.layers.SeparableConv1D(128, kernel_size=3, activation="relu", padding="same")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs)

model = build_model_v2()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)
model.summary()

In [ ]:
history_v2 = model.fit(
    train_ds_norm,
    validation_data=test_ds_norm,
    epochs=15,
    class_weight={0: 1.0, 1: 5.0}
)

In [ ]:
print("Final training metrics:", {k: v[-1] for k, v in history_v2.history.items() if not k.startswith("val_")})
print("Final validation metrics:", {k: v[-1] for k, v in history_v2.history.items() if k.startswith("val_")})

In [ ]:
y_true_full_v2, y_pred_probs_full_v2 = [], []
for x_batch, y_batch in full_test_ds_norm:
    preds = model.predict(x_batch, verbose=0)
    y_true_full_v2.extend(y_batch.numpy())
    y_pred_probs_full_v2.extend(preds.flatten())

total_hours_full = len(y_true_full_v2) * 4 / 3600
for min_c in [1, 5, 10, 15]:
    y_pred_smoothed = smooth_predictions(y_pred_probs_full_v2, threshold=0.3, min_consecutive=min_c)
    p = precision_score(y_true_full_v2, y_pred_smoothed)
    r = recall_score(y_true_full_v2, y_pred_smoothed)
    fp = np.sum((y_pred_smoothed == 1) & (np.array(y_true_full_v2) == 0))
    fa_hr = fp / total_hours_full
    print(f"min_consecutive={min_c}: Precision={p:.3f}, Recall={r:.3f}, FA/hr={fa_hr:.2f}")

In [ ]:
from tensorflow.keras import regularizers

def build_model_v3(input_shape=(18, 1024)):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2, 1))(inputs)

    x = tf.keras.layers.SeparableConv1D(
        16, kernel_size=7, activation="relu", padding="same",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)

    x = tf.keras.layers.SeparableConv1D(
        32, kernel_size=5, activation="relu", padding="same",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)

    x = tf.keras.layers.SeparableConv1D(
        64, kernel_size=3, activation="relu", padding="same",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs)

model = build_model_v3()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)
model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=3, restore_best_weights=True
)

history_v3 = model.fit(
    train_ds_norm,
    validation_data=test_ds_norm,
    epochs=25,  # early stopping will cut this short if needed
    class_weight={0: 1.0, 1: 2.0},  # gentler than 5.0
    callbacks=[early_stop]
)

In [ ]:
from tensorflow.keras import regularizers

def build_model_v3(input_shape=(18, 1024)):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2, 1))(inputs)

    x = tf.keras.layers.SeparableConv1D(
        16, kernel_size=7, activation="relu", padding="same",
        depthwise_regularizer=regularizers.l2(1e-4),
        pointwise_regularizer=regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)

    x = tf.keras.layers.SeparableConv1D(
        32, kernel_size=5, activation="relu", padding="same",
        depthwise_regularizer=regularizers.l2(1e-4),
        pointwise_regularizer=regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)

    x = tf.keras.layers.SeparableConv1D(
        64, kernel_size=3, activation="relu", padding="same",
        depthwise_regularizer=regularizers.l2(1e-4),
        pointwise_regularizer=regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    x = tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    return tf.keras.Model(inputs, outputs)

model = build_model_v3()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)
model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=3, restore_best_weights=True
)

history_v3 = model.fit(
    train_ds_norm,
    validation_data=test_ds_norm,
    epochs=25,
    class_weight={0: 1.0, 1: 2.0},
    callbacks=[early_stop]
)

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import glob
import shutil
from collections import defaultdict
import tensorflow as tf
from tensorflow.keras import regularizers

!pip install pyedflib --quiet
import pyedflib

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/chbmit"

# Reload annotation/window CSVs
df = pd.read_csv(os.path.join(DRIVE_ROOT, "master_annotations.csv"))
window_df = pd.read_csv(os.path.join(DRIVE_ROOT, "window_index.csv"))
train_subset = pd.read_csv(os.path.join(DRIVE_ROOT, "train_subset.csv"))

# Common channel set
COMMON_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ"
]

# Copy rebalanced train shards locally
LOCAL_SHARD_DIR = "/content/local_shards"
os.makedirs(LOCAL_SHARD_DIR, exist_ok=True)
drive_shard_dir = os.path.join(DRIVE_ROOT, "shards")
for fname in os.listdir(drive_shard_dir):
    src, dst = os.path.join(drive_shard_dir, fname), os.path.join(LOCAL_SHARD_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

subject_shards = defaultdict(list)
for fname in os.listdir(LOCAL_SHARD_DIR):
    subject_shards[fname.split("_")[0]].append(os.path.join(LOCAL_SHARD_DIR, fname))

# Copy full (honest) test shards locally
TEST_SHARD_DIR = os.path.join(DRIVE_ROOT, "test_shards_full")
LOCAL_TEST_SHARD_DIR = "/content/local_test_shards_full"
os.makedirs(LOCAL_TEST_SHARD_DIR, exist_ok=True)
for fname in os.listdir(TEST_SHARD_DIR):
    src, dst = os.path.join(TEST_SHARD_DIR, fname), os.path.join(LOCAL_TEST_SHARD_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

full_test_shards = defaultdict(list)
for fname in os.listdir(LOCAL_TEST_SHARD_DIR):
    full_test_shards[fname.split("_")[0]].append(os.path.join(LOCAL_TEST_SHARD_DIR, fname))

# Recreate the exact same train/test subject split (fixed seed)
subjects = sorted(subject_shards.keys())
np.random.seed(42)
shuffled = np.random.permutation(subjects)
n_test_subjects = max(1, int(0.2 * len(subjects)))
test_subjects = set(shuffled[:n_test_subjects])
train_subjects = set(shuffled[n_test_subjects:])

# Re-define helper functions
def normalize_window(window):
    mean = window.mean(axis=1, keepdims=True)
    std = window.std(axis=1, keepdims=True) + 1e-8
    return (window - mean) / std

def make_dataset_normalized(subject_list, subject_shards_map, batch_size=64, shuffle=True):
    def gen():
        for subject in subject_list:
            for path in subject_shards_map[subject]:
                data = np.load(path)
                signals = data["signals"].astype(np.float32)
                labels = data["labels"].astype(np.float32)
                for sig, lbl in zip(signals, labels):
                    yield normalize_window(sig), lbl
    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(18, 1024), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=2000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def smooth_predictions(probs, threshold=0.3, min_consecutive=3):
    binary = (np.array(probs) > threshold).astype(int)
    smoothed = np.zeros_like(binary)
    count = 0
    for i in range(len(binary)):
        count = count + 1 if binary[i] == 1 else 0
        if count >= min_consecutive:
            smoothed[max(0, i - min_consecutive + 1):i + 1] = 1
    return smoothed

train_ds_norm = make_dataset_normalized(list(train_subjects), subject_shards, shuffle=True)
test_ds_norm = make_dataset_normalized(list(test_subjects), subject_shards, shuffle=False)
full_test_ds_norm = make_dataset_normalized(list(test_subjects), full_test_shards, shuffle=False)

print("Recovery complete. Ready to rebuild model and retrain.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 36.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


OSError: [Errno 107] Transport endpoint is not connected

In [ ]:
import time

def robust_copy_dir(src_dir, dst_dir, max_retries=3):
    os.makedirs(dst_dir, exist_ok=True)
    failed = []
    for fname in os.listdir(src_dir):
        src, dst = os.path.join(src_dir, fname), os.path.join(dst_dir, fname)
        if os.path.exists(dst):
            continue
        success = False
        for attempt in range(max_retries):
            try:
                shutil.copy(src, dst)
                success = True
                break
            except OSError:
                if attempt < max_retries - 1:
                    drive.mount('/content/drive', force_remount=True)
                    time.sleep(2)
        if not success:
            failed.append(fname)
    return failed

# Copy train shards
failed_train = robust_copy_dir(drive_shard_dir, LOCAL_SHARD_DIR)
print(f"Train shards — failed: {len(failed_train)}")

# Copy full test shards
failed_test = robust_copy_dir(TEST_SHARD_DIR, LOCAL_TEST_SHARD_DIR)
print(f"Test shards — failed: {len(failed_test)}")

Train shards — failed: 0


NameError: name 'TEST_SHARD_DIR' is not defined

In [ ]:
TEST_SHARD_DIR = os.path.join(DRIVE_ROOT, "test_shards_full")
LOCAL_TEST_SHARD_DIR = "/content/local_test_shards_full"

failed_test = robust_copy_dir(TEST_SHARD_DIR, LOCAL_TEST_SHARD_DIR)
print(f"Test shards — failed: {len(failed_test)}")

Test shards — failed: 0


In [ ]:
# Rebuild subject -> shard path maps
subject_shards = defaultdict(list)
for fname in os.listdir(LOCAL_SHARD_DIR):
    subject_shards[fname.split("_")[0]].append(os.path.join(LOCAL_SHARD_DIR, fname))

full_test_shards = defaultdict(list)
for fname in os.listdir(LOCAL_TEST_SHARD_DIR):
    full_test_shards[fname.split("_")[0]].append(os.path.join(LOCAL_TEST_SHARD_DIR, fname))

# Recreate the exact same train/test split (fixed seed — same result every time)
subjects = sorted(subject_shards.keys())
np.random.seed(42)
shuffled = np.random.permutation(subjects)
n_test_subjects = max(1, int(0.2 * len(subjects)))
test_subjects = set(shuffled[:n_test_subjects])
train_subjects = set(shuffled[n_test_subjects:])

print(f"Train subjects ({len(train_subjects)}): {sorted(train_subjects)}")
print(f"Test subjects ({len(test_subjects)}): {sorted(test_subjects)}")

# Helper functions
def normalize_window(window):
    mean = window.mean(axis=1, keepdims=True)
    std = window.std(axis=1, keepdims=True) + 1e-8
    return (window - mean) / std

def make_dataset_normalized(subject_list, subject_shards_map, batch_size=64, shuffle=True):
    def gen():
        for subject in subject_list:
            for path in subject_shards_map[subject]:
                data = np.load(path)
                signals = data["signals"].astype(np.float32)
                labels = data["labels"].astype(np.float32)
                for sig, lbl in zip(signals, labels):
                    yield normalize_window(sig), lbl
    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(18, 1024), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=2000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def smooth_predictions(probs, threshold=0.3, min_consecutive=3):
    binary = (np.array(probs) > threshold).astype(int)
    smoothed = np.zeros_like(binary)
    count = 0
    for i in range(len(binary)):
        count = count + 1 if binary[i] == 1 else 0
        if count >= min_consecutive:
            smoothed[max(0, i - min_consecutive + 1):i + 1] = 1
    return smoothed

train_ds_norm = make_dataset_normalized(list(train_subjects), subject_shards, shuffle=True)
test_ds_norm = make_dataset_normalized(list(test_subjects), subject_shards, shuffle=False)
full_test_ds_norm = make_dataset_normalized(list(test_subjects), full_test_shards, shuffle=False)

print("Pipeline fully rebuilt and ready.")

Train subjects (20): [np.str_('chb02'), np.str_('chb03'), np.str_('chb04'), np.str_('chb05'), np.str_('chb06'), np.str_('chb07'), np.str_('chb08'), np.str_('chb10'), np.str_('chb11'), np.str_('chb12'), np.str_('chb13'), np.str_('chb14'), np.str_('chb15'), np.str_('chb16'), np.str_('chb18'), np.str_('chb20'), np.str_('chb21'), np.str_('chb22'), np.str_('chb23'), np.str_('chb24')]
Test subjects (4): [np.str_('chb01'), np.str_('chb09'), np.str_('chb17'), np.str_('chb19')]
Pipeline fully rebuilt and ready.


In [ ]:
from tensorflow.keras import regularizers

def build_model_v3(input_shape=(18, 1024)):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Permute((2, 1))(inputs)
    x = tf.keras.layers.SeparableConv1D(16, 7, activation="relu", padding="same",
        depthwise_regularizer=regularizers.l2(1e-4), pointwise_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.SeparableConv1D(32, 5, activation="relu", padding="same",
        depthwise_regularizer=regularizers.l2(1e-4), pointwise_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(4)(x)
    x = tf.keras.layers.SeparableConv1D(64, 3, activation="relu", padding="same",
        depthwise_regularizer=regularizers.l2(1e-4), pointwise_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inputs, outputs)

model = build_model_v3()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True)

history_v3 = model.fit(
    train_ds_norm, validation_data=test_ds_norm, epochs=25,
    class_weight={0: 1.0, 1: 2.0}, callbacks=[early_stop]
)

Epoch 1/25
   1273/Unknown 126s 85ms/step - accuracy: 0.9503 - auc: 0.8412 - loss: 0.2772 - precision: 0.5990 - recall: 0.3994

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) UNKNOWN:  BadZipFile: File is not a zip file
Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)

  File "/usr/local/lib/python3.13/dist-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "/usr/local/lib/python3.13/dist-packages/tensorflow/python/data/ops/from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "/tmp/ipykernel_2325/2290082160.py", line 31, in gen
    data = np.load(path)

  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 476, in load
    ret = NpzFile(fid, own_fid=o

In [ ]:
def validate_and_repair_shards(local_dir, drive_dir):
    corrupted = []
    for fname in os.listdir(local_dir):
        path = os.path.join(local_dir, fname)
        try:
            data = np.load(path)
            _ = data["signals"].shape  # force actual read, not just open
        except Exception:
            corrupted.append(fname)

    print(f"Found {len(corrupted)} corrupted shards out of {len(os.listdir(local_dir))}")

    for fname in corrupted:
        local_path = os.path.join(local_dir, fname)
        drive_path = os.path.join(drive_dir, fname)
        os.remove(local_path)  # delete the bad local copy
        shutil.copy(drive_path, local_path)  # force fresh copy from Drive

    print(f"Repaired {len(corrupted)} shards")
    return corrupted

# Check and repair both shard sets
corrupted_train = validate_and_repair_shards(LOCAL_SHARD_DIR, drive_shard_dir)
corrupted_test = validate_and_repair_shards(LOCAL_TEST_SHARD_DIR, TEST_SHARD_DIR)

print(f"Train corrupted/repaired: {len(corrupted_train)}")
print(f"Test corrupted/repaired: {len(corrupted_test)}")

Found 1 corrupted shards out of 673
Repaired 1 shards
Found 0 corrupted shards out of 112
Repaired 0 shards
Train corrupted/repaired: 1
Test corrupted/repaired: 0


In [ ]:
model = build_model_v3()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall"), tf.keras.metrics.AUC(name="auc")]
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True)

history_v3 = model.fit(
    train_ds_norm, validation_data=test_ds_norm, epochs=25,
    class_weight={0: 1.0, 1: 2.0}, callbacks=[early_stop]
)

Epoch 1/25
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 132s 95ms/step - accuracy: 0.9556 - auc: 0.9152 - loss: 0.2274 - precision: 0.6789 - recall: 0.5758 - val_accuracy: 0.8940 - val_auc: 0.7567 - val_loss: 0.2801 - val_precision: 0.1785 - val_recall: 0.4367
Epoch 2/25
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 117s 90ms/step - accuracy: 0.9624 - auc: 0.9346 - loss: 0.1931 - precision: 0.7406 - recall: 0.6324 - val_accuracy: 0.9071 - val_auc: 0.7420 - val_loss: 0.2644 - val_precision: 0.1781 - val_recall: 0.3472
Epoch 3/25
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 115s 89ms/step - accuracy: 0.9683 - auc: 0.9479 - loss: 0.1706 - precision: 0.7900 - recall: 0.6859 - val_accuracy: 0.8713 - val_auc: 0.7465 - val_loss: 0.3310 - val_precision: 0.1285 - val_recall: 0.3673
Epoch 4/25
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 117s 91ms/step - accuracy: 0.9704 - auc: 0.9570 - loss: 0.1556 - precision: 0.7973 - recall: 0.7175 - val_accuracy: 0.8201 - val_auc: 0.7590 - val_loss: 0.4272 - val_precision: 0.1065 - val_recall: 0.4552
Epoch 5/25
1

In [ ]:
print("Final training metrics:", {k: v[-1] for k, v in history_v3.history.items() if not k.startswith("val_")})
print("Final validation metrics:", {k: v[-1] for k, v in history_v3.history.items() if k.startswith("val_")})
print(f"Stopped at epoch: {len(history_v3.history['loss'])}")
print(f"Best val_auc epoch: {np.argmax(history_v3.history['val_auc']) + 1}")

Final training metrics: {'accuracy': 0.9735488295555115, 'auc': 0.9687264561653137, 'loss': 0.13460586965084076, 'precision': 0.8115645051002502, 'recall': 0.7621798515319824}
Final validation metrics: {'val_accuracy': 0.7375985980033875, 'val_auc': 0.7395289540290833, 'val_loss': 0.7245123386383057, 'val_precision': 0.08595237880945206, 'val_recall': 0.5570987462997437}
Stopped at epoch: 7
Best val_auc epoch: 4


In [ ]:
from scipy import signal as scipy_signal

def compute_spectrogram(window, fs=256):
    # window shape: (18 channels, 1024 samples)
    specs = []
    for ch in range(window.shape[0]):
        f, t, Sxx = scipy_signal.spectrogram(window[ch], fs=fs, nperseg=64, noverlap=32)
        specs.append(np.log(Sxx + 1e-8))  # log-power, standard practice
    return np.array(specs)  # shape: (18, freq_bins, time_bins)


In [ ]:
from scipy import signal as scipy_signal

FS = 256

def compute_spectrogram(window, fs=FS, nperseg=64, noverlap=32):
    """window: (18, 1024) raw -> (freq_bins, time_bins, 18) log-power"""
    specs = []
    for ch in range(window.shape[0]):
        f, t, Sxx = scipy_signal.spectrogram(
            window[ch], fs=fs, nperseg=nperseg, noverlap=noverlap
        )
        specs.append(np.log(Sxx + 1e-8))
    specs = np.stack(specs, axis=-1)          # (freq_bins, time_bins, 18)
    # per-channel z-score across the time-frequency plane
    mean = specs.mean(axis=(0, 1), keepdims=True)
    std = specs.std(axis=(0, 1), keepdims=True) + 1e-8
    return ((specs - mean) / std).astype(np.float32)

# Confirm the output shape before wiring it into the pipeline
sample_path = subject_shards[sorted(train_subjects)[0]][0]
sample = np.load(sample_path)["signals"][0].astype(np.float32)
SPEC_SHAPE = compute_spectrogram(sample).shape
print(f"Spectrogram shape per window: {SPEC_SHAPE}")

Spectrogram shape per window: (33, 31, 18)


In [ ]:
def make_spec_dataset(subject_list, shards_map, batch_size=64, shuffle=True):
    def gen():
        for subject in subject_list:
            for path in shards_map[subject]:
                data = np.load(path)
                signals = data["signals"].astype(np.float32)
                labels = data["labels"].astype(np.float32)
                for sig, lbl in zip(signals, labels):
                    yield compute_spectrogram(sig), lbl

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=SPEC_SHAPE, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32),
        ),
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=2000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_spec = make_spec_dataset(list(train_subjects), subject_shards, shuffle=True)
test_ds_spec = make_spec_dataset(list(test_subjects), subject_shards, shuffle=False)
full_test_ds_spec = make_spec_dataset(list(test_subjects), full_test_shards, shuffle=False)

for xb, yb in train_ds_spec.take(1):
    print("Batch:", xb.shape, yb.shape)

Batch: (64, 33, 31, 18) (64,)


In [ ]:
def build_spec_model(input_shape=SPEC_SHAPE):
    inputs = tf.keras.Input(shape=input_shape)

    x = tf.keras.layers.SeparableConv2D(16, 3, padding="same", activation="relu",
        depthwise_regularizer=regularizers.l2(1e-4),
        pointwise_regularizer=regularizers.l2(1e-4))(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.SeparableConv2D(32, 3, padding="same", activation="relu",
        depthwise_regularizer=regularizers.l2(1e-4),
        pointwise_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.SeparableConv2D(64, 3, padding="same", activation="relu",
        depthwise_regularizer=regularizers.l2(1e-4),
        pointwise_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(32, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    return tf.keras.Model(inputs, outputs)

spec_model = build_spec_model()
spec_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall"),
             tf.keras.metrics.AUC(name="auc")],
)
spec_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 33, 31, 18)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 33, 31, 16)     │           466 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 33, 31, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 15, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_1              │ (None, 16, 15, 32)     │           688 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 16, 15, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_2              │ (None, 8, 7, 64)       │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 8, 7, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,115 (23.89 KB)

 Trainable params: 5,891 (23.01 KB)

 Non-trainable params: 224 (896.00 B)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=4, restore_best_weights=True
)

history_spec = spec_model.fit(
    train_ds_spec, validation_data=test_ds_spec, epochs=25,
    class_weight={0: 1.0, 1: 3.0}, callbacks=[early_stop]
)

print("Train:", {k: v[-1] for k, v in history_spec.history.items() if not k.startswith("val_")})
print("Val:  ", {k: v[-1] for k, v in history_spec.history.items() if k.startswith("val_")})
print("Best val_auc epoch:", int(np.argmax(history_spec.history["val_auc"])) + 1)

Epoch 1/25
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 750s 567ms/step - accuracy: 0.9374 - auc: 0.9051 - loss: 0.3161 - precision: 0.5078 - recall: 0.5858 - val_accuracy: 0.6840 - val_auc: 0.7543 - val_loss: 0.6968 - val_precision: 0.0851 - val_recall: 0.6836
Epoch 2/25
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 764s 545ms/step - accuracy: 0.9500 - auc: 0.9330 - loss: 0.2624 - precision: 0.5954 - recall: 0.6734 - val_accuracy: 0.9197 - val_auc: 0.6994 - val_loss: 0.2574 - val_precision: 0.1200 - val_recall: 0.1497
Epoch 3/25
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 694s 534ms/step - accuracy: 0.9555 - auc: 0.9454 - loss: 0.2389 - precision: 0.6339 - recall: 0.7148 - val_accuracy: 0.7952 - val_auc: 0.7476 - val_loss: 0.4403 - val_precision: 0.1018 - val_recall: 0.5077
Epoch 4/25
1273/1273 ━━━━━━━━━━━━━━━━━━━━ 717s 552ms/step - accuracy: 0.9550 - auc: 0.9468 - loss: 0.2353 - precision: 0.6290 - recall: 0.7163 - val_accuracy: 0.7756 - val_auc: 0.7528 - val_loss: 0.4489 - val_precision: 0.1022 - val_recall: 0.5710
Epoch 5/